# Core JS

> Master IIFE composer for the token selector JS runtime.

In [ ]:
#| default_exp js.core

In [ ]:
#| export
from typing import Any, Tuple

from fasthtml.common import Script

from cjm_fasthtml_token_selector.core.config import TokenSelectorConfig
from cjm_fasthtml_token_selector.core.models import TokenSelectorState
from cjm_fasthtml_token_selector.core.html_ids import TokenSelectorHtmlIds
from cjm_fasthtml_token_selector.js.navigation import generate_navigation_js
from cjm_fasthtml_token_selector.js.display import generate_display_js
from cjm_fasthtml_token_selector.js.repeat import generate_key_repeat_js

## Global Callback Names

In [ ]:
#| export
_GLOBAL_CALLBACKS = (
    "activate",
    "deactivate",
    "selectGap",
    "selectWord",
    "selectToken",
    "moveToFirst",
    "moveToLast",
)

def global_callback_name(
    prefix:str,    # token selector instance prefix
    callback:str,  # base callback name
) -> str:  # global function name
    """Generate a prefix-unique global callback name."""
    return f"{prefix}_{callback}"

In [ ]:
assert global_callback_name("ts0", "activate") == "ts0_activate"
assert global_callback_name("my-split", "deactivate") == "my-split_deactivate"
print("global_callback_name tests passed!")

global_callback_name tests passed!


## IIFE Sub-Generators

In [ ]:
#| export
def _generate_state_init_js(
    config:TokenSelectorConfig,  # config for this instance
    state:TokenSelectorState,    # initial state
) -> str:  # JS code fragment
    """Generate state initialization code."""
    return f"""
    // State
    ns.anchor = {state.anchor};
    ns.focus = {state.focus};
    ns.wordCount = {state.word_count};
    ns.active = false;
    ns.mode = '{config.selection_mode}';
    ns._prevAnchor = -1;
"""

In [ ]:
#| export
def _generate_on_change_js(
    config:TokenSelectorConfig,  # config for this instance
) -> str:  # JS code fragment
    """Generate the on-change callback dispatcher."""
    if config.on_change_callback:
        return f"""
    ns._fireOnChange = function() {{
        if (typeof window['{config.on_change_callback}'] === 'function') {{
            window['{config.on_change_callback}'](ns.anchor, ns.focus, ns.mode);
        }}
    }};
"""
    return """
    ns._fireOnChange = function() {};
"""

In [ ]:
#| export
def _generate_activation_js(
    config:TokenSelectorConfig,  # config for this instance
    ids:TokenSelectorHtmlIds,    # HTML IDs
) -> str:  # JS code fragment
    """Generate activate/deactivate functions."""
    return f"""
    ns.activate = function() {{
        ns.active = true;
        // Re-read word count from DOM
        var grid = document.getElementById('{ids.token_grid}');
        if (grid) ns.wordCount = parseInt(grid.dataset.wordCount) || 0;
        // Reset position
        ns.anchor = 0;
        ns.focus = 0;
        // Add key listeners
        document.addEventListener('keydown', ns._handleKeyDown, true);
        document.addEventListener('keyup', ns._handleKeyUp, true);
        ns.updateDisplay();
        ns.updateInputs();
    }};

    ns.deactivate = function() {{
        ns.active = false;
        // Cancel repeat
        if (ns._repeatTimer) {{
            clearTimeout(ns._repeatTimer);
            ns._repeatTimer = null;
        }}
        ns._repeatKey = null;
        ns._repeatShift = false;
        // Remove key listeners
        document.removeEventListener('keydown', ns._handleKeyDown, true);
        document.removeEventListener('keyup', ns._handleKeyUp, true);
    }};
"""

In [ ]:
#| export
def _generate_global_callbacks_js(
    config:TokenSelectorConfig,  # config for this instance
) -> str:  # JS code fragment
    """Generate global callback wrappers."""
    lines = []
    for cb_name in _GLOBAL_CALLBACKS:
        global_name = global_callback_name(config.prefix, cb_name)
        lines.append(
            f"    window['{global_name}'] = function() {{ "
            f"if (ns.{cb_name}) ns.{cb_name}.apply(ns, arguments); }};"
        )
    return "\n".join(lines)

In [ ]:
#| export
def _generate_settle_handler_js(
    config:TokenSelectorConfig,  # config for this instance
    ids:TokenSelectorHtmlIds,    # HTML IDs
) -> str:  # JS code fragment
    """Generate htmx:afterSettle handler for swap resilience."""
    guard = f"_tsMasterListener_{config.prefix}"
    return f"""
    // HTMX afterSettle: re-read word count and update display
    if (!window['{guard}']) {{
        window['{guard}'] = true;
        document.body.addEventListener('htmx:afterSettle', function() {{
            var grid = document.getElementById('{ids.token_grid}');
            if (grid && ns.active) {{
                ns.wordCount = parseInt(grid.dataset.wordCount) || 0;
                ns.updateDisplay();
                ns.updateInputs();
            }}
        }});
    }}
"""

## Master Composer

In [ ]:
#| export
def generate_token_selector_js(
    config:TokenSelectorConfig,              # config for this instance
    ids:TokenSelectorHtmlIds,                # HTML IDs
    state:TokenSelectorState = None,         # initial state
    extra_scripts:Tuple[str, ...] = (),      # additional JS to include in the IIFE
) -> Any:  # Script element with the complete IIFE
    """Compose all token selector JS into a single namespaced IIFE."""
    if state is None:
        state = TokenSelectorState()

    # Collect all JS fragments
    fragments = [
        _generate_state_init_js(config, state),
        _generate_on_change_js(config),
        generate_display_js(config, ids),
        generate_navigation_js(config, ids),
        generate_key_repeat_js(config),
        _generate_activation_js(config, ids),
        _generate_global_callbacks_js(config),
        _generate_settle_handler_js(config, ids),
    ]

    # Consumer extra scripts
    if extra_scripts:
        fragments.append("\n".join(extra_scripts))

    body = "\n".join(fragments)
    prefix = config.prefix

    iife = f"""(function() {{
    'use strict';
    window.tokenSelectors = window.tokenSelectors || {{}};
    var ns = window.tokenSelectors['{prefix}'] = {{}};
{body}
}})();"""

    return Script(iife)

In [ ]:
from cjm_fasthtml_token_selector.core.config import TokenSelectorConfig, _reset_prefix_counter
from cjm_fasthtml_token_selector.core.models import TokenSelectorState
from cjm_fasthtml_token_selector.core.html_ids import TokenSelectorHtmlIds

_reset_prefix_counter()

cfg = TokenSelectorConfig(prefix="ts0")
ids = TokenSelectorHtmlIds(prefix="ts0")
state = TokenSelectorState(anchor=3, focus=3, word_count=10)

script = generate_token_selector_js(cfg, ids, state)
assert script is not None

# Verify IIFE structure via string representation
js_str = str(script)
assert "window.tokenSelectors" in js_str
assert "ts0" in js_str
assert "ns.activate" in js_str
assert "ns.deactivate" in js_str
assert "ns.moveLeft" in js_str
assert "ns._handleKeyDown" in js_str
assert "ts0_activate" in js_str
assert "_tsMasterListener_ts0" in js_str
print("IIFE structure verified!")

# With extra scripts
script2 = generate_token_selector_js(cfg, ids, state, extra_scripts=("ns.customFn = function() {};",))
assert "ns.customFn" in str(script2)
print("Extra scripts injection verified!")

# Multi-instance: different prefixes don't collide
cfg2 = TokenSelectorConfig(prefix="ts1")
ids2 = TokenSelectorHtmlIds(prefix="ts1")
script3 = generate_token_selector_js(cfg2, ids2)
js_str3 = str(script3)
assert "ts1" in js_str3
assert "ts0" not in js_str3
print("Multi-instance isolation verified!")

# Default state when none provided
script4 = generate_token_selector_js(cfg, ids)
assert script4 is not None
print("Default state fallback verified!")

_reset_prefix_counter()
print("All JS core tests passed!")

IIFE structure verified!
Extra scripts injection verified!
Multi-instance isolation verified!
Default state fallback verified!
All JS core tests passed!


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()